# Frost observations ingestion

Download raw monthly `air_temperature` observations for 2024–2025. Run after `frost_series.py`

In [1]:
from bronze_layer.frost_observations import ingest, eligible_stations

/home/roland/Desktop/DAT535_DEV2/group14/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Check which stations are eligible

Eligibility comes from the cached Frost available-series metadata.


In [2]:
stations = eligible_stations("2024-01-01", "2026-01-01")
print("Eligible stations:", len(stations))
print("First 10:", stations[:10])

Eligible stations: 1121
First 10: ['SN10055', 'SN10070', 'SN10380', 'SN1070', 'SN10800', 'SN11000', 'SN11060', 'SN11125', 'SN11340', 'SN1135']


## Small test: 3 stations and 2 months

In [3]:
test_result = ingest(
    start="2024-01-01",
    end="2024-03-01",
    station_ids=stations[:3],
)
print({k: v for k, v in test_result.items() if k.endswith("_count") or k == "total_station_months"})
print("Failures:", test_result["failed"][:5])


Frost observations: 100%|██████████| 6/6 [00:05<00:00,  1.18month/s, cached=0, downloaded=4, empty=0, failed=2]

{'station_count': 3, 'total_station_months': 6, 'downloaded_count': 4, 'cached_count': 0, 'empty_count': 0, 'failed_count': 2}
Failures: [{'station': 'SN10055', 'start': '2024-01-01', 'end': '2024-02-01', 'error': '412 Client Error: Precondition Failed for url: https://frost.met.no/observations/v0.jsonld?sources=SN10055%3Aall&referencetime=2024-01-01%2F2024-02-01&elements=air_temperature'}, {'station': 'SN10055', 'start': '2024-02-01', 'end': '2024-03-01', 'error': '412 Client Error: Precondition Failed for url: https://frost.met.no/observations/v0.jsonld?sources=SN10055%3Aall&referencetime=2024-02-01%2F2024-03-01&elements=air_temperature'}]


## Inspect one saved response

In [4]:
import json
from bronze_layer.frost_observations import DEFAULT_ROOT
files = sorted(DEFAULT_ROOT.glob("station=*/period=*/response.json"))
if files:
    sample = json.loads(files[0].read_text(encoding="utf-8"))
    print("File:", files[0])
    print("Number of data entries:", len(sample.get("data", [])))
    print(json.dumps(sample.get("data", [])[:1], indent=2, ensure_ascii=False)[:3000])
else:
    print("No observation files yet")


File: /home/roland/Desktop/DAT535_DEV2/group14/data/bronze/frost/observations/station=SN10070/period=2024-01-01_to_2024-02-01/response.json
Number of data entries: 4433
[
  {
    "sourceId": "SN10070:0",
    "referenceTime": "2024-01-01T00:00:00.000Z",
    "observations": [
      {
        "elementId": "air_temperature",
        "value": -9.1,
        "unit": "degC",
        "level": {
          "levelType": "height_above_ground",
          "unit": "m",
          "value": 2
        },
        "timeOffset": "PT0H",
        "timeResolution": "PT10M",
        "timeSeriesId": 0,
        "performanceCategory": "C",
        "exposureCategory": "4",
        "qualityCode": 0
      }
    ]
  }
]


## Full run: all eligible stations 2024 - 2025

In [ ]:
full_result = ingest(start="2024-01-01", end="2026-01-01")
print({k: v for k, v in full_result.items() if k.endswith("_count") or k == "total_station_months"})
print("Failed months:", full_result["failed"][:10])